### Coding with Modular Python

In [ ]:
import torch
import torch.nn
from tqdm.auto import tqdm
from timeit import default_timer
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.transforms import ToTensor



c:\Users\aryan\PycharmProjects\Independent_Study\Independent_Study\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_dataset(name, transform):
    if name == 'cifar10':
        train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
        test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    elif name == 'mnist':
        train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    elif name == 'emnist':
        train = datasets.EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
        test = datasets.EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)
    
    print(f"Loaded {name}: {len(train)} train, {len(test)} test")
    return train, test

In [ ]:
def strategy_baseline(num_classes=10):
    return BaseCNN(num_classes=num_classes)


def strategy_feature_extraction(num_classes=10):
    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def strategy_full_finetune(num_classes=10):
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def strategy_gradual_unfreeze(train_loader, test_loader, num_classes=10):
    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)

    stages = [
        ("fc only",  [],              0.001,  3),
        ("+ layer4", [model.layer4],   0.0005, 3),
        ("+ layer3", [model.layer3],   0.0003, 3),
        ("+ layer2", [model.layer2],   0.0001, 2),
    ]

    loss_fn = nn.CrossEntropyLoss()

    for stage_name, layers_to_unfreeze, lr, epochs in stages:
        for layer in layers_to_unfreeze:
            for param in layer.parameters():
                param.requires_grad = True

        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr
        )

        model.train()
        for epoch in range(epochs):
            running_loss = 0
            for X, y in train_loader:
                X, y = X.to(device), y.to(device)
                optimizer.zero_grad()
                output = model(X)
                loss = loss_fn(output, y)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
            print(f"  {stage_name} - Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

    acc = test_model(model, test_loader)
    return acc